In [1]:
import torch
import os
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("No GPU detected. We can still proceed with CPU inference.")

Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 15.64


In [2]:
import subprocess

result = subprocess.run(
    ["nvcc", "--version"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("CUDA compiler (nvcc) is not available.")
    print(result.stderr)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0



In [3]:
import os

os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
os.environ["FORCE_CMAKE"] = "1"

!pip install --no-cache-dir llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 MB 125.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Operation cancelled by user


In [5]:
import requests

url = "https://api.github.com/repos/ggml-org/llama.cpp/releases"
response = requests.get(url, params={"per_page": 10})

print("HTTP status:", response.status_code)
response.raise_for_status()

releases = response.json()

for r in releases:
    cuda_assets = [
        a for a in r.get("assets", [])
        if "cuda" in a["name"].lower()
        and "ubuntu" in a["name"].lower()
    ]

    if cuda_assets:
        print("\nRelease:", r["tag_name"])
        for a in cuda_assets:
            print("Asset:", a["name"])
            print("Size (MB):", round(a["size"] / (1024**2), 1))
            print("URL:", a["browser_download_url"])

HTTP status: 200

Release: b12542
Asset: cudart-llama-b12542-bin-ubuntu-cuda-12.8-x64.tar.gz
Size (MB): 566.8
URL: https://github.com/ggml-org/llama.cpp/releases/download/b12542/cudart-llama-b12542-bin-ubuntu-cuda-12.8-x64.tar.gz
Asset: cudart-llama-b12542-bin-ubuntu-cuda-13.5-arm64.tar.gz
Size (MB): 533.9
URL: https://github.com/ggml-org/llama.cpp/releases/download/b12542/cudart-llama-b12542-bin-ubuntu-cuda-13.5-arm64.tar.gz
Asset: cudart-llama-b12542-bin-ubuntu-cuda-13.5-x64.tar.gz
Size (MB): 423.5
URL: https://github.com/ggml-org/llama.cpp/releases/download/b12542/cudart-llama-b12542-bin-ubuntu-cuda-13.5-x64.tar.gz
Asset: llama-b12542-bin-ubuntu-cuda-12.8-x64.tar.gz
Size (MB): 212.6
URL: https://github.com/ggml-org/llama.cpp/releases/download/b12542/llama-b12542-bin-ubuntu-cuda-12.8-x64.tar.gz
Asset: llama-b12542-bin-ubuntu-cuda-13.5-arm64.tar.gz
Size (MB): 212.6
URL: https://github.com/ggml-org/llama.cpp/releases/download/b12542/llama-b12542-bin-ubuntu-cuda-13.5-arm64.tar.gz
Asset:

In [6]:
import os

url = "https://github.com/ggml-org/llama.cpp/releases/download/b12542/cudart-llama-b12542-bin-ubuntu-cuda-12.8-x64.tar.gz"

!wget -q --show-progress -O llama-cpp.tar.gz "$url"

llama-cpp.tar.gz    100%[===================>] 566.84M  16.7MB/s    in 34s     


In [7]:
!mkdir -p llama-cpp
!tar -xzf llama-cpp.tar.gz -C llama-cpp

In [8]:
!find llama-cpp -type f -name "llama-cli"

In [9]:
import os

print("Extraction folder exists:", os.path.exists("llama-cpp"))

if os.path.exists("llama-cpp"):
    for root, dirs, files in os.walk("llama-cpp"):
        for file in files:
            if file == "llama-cli":
                print("Found executable:", os.path.join(root, file))

Extraction folder exists: True


In [10]:
import os

for root, dirs, files in os.walk("llama-cpp"):
    level = root.replace("llama-cpp", "").count(os.sep)
    indent = " " * 2 * level
    print(indent + os.path.basename(root) + "/")

    for file in files[:10]:
        print(indent + "  " + file)

llama-cpp/
  cudart-llama-b12542-bin-ubuntu-cuda-12.8-x64/
    libcublasLt.so.12
    libcublas.so.12
    libcudart.so.12


In [11]:
url = "https://github.com/ggml-org/llama.cpp/releases/download/b12542/llama-b12542-bin-ubuntu-cuda-12.8-x64.tar.gz"

!wget -q --show-progress -O llama-binaries.tar.gz "$url"

llama-binaries.tar. 100%[===================>] 212.60M  42.6MB/s    in 4.6s    


In [12]:
!mkdir -p llama-binaries
!tar -xzf llama-binaries.tar.gz -C llama-binaries

In [13]:
!find llama-binaries -type f -name "llama-cli"

llama-binaries/llama-b12542/llama-cli


In [14]:
import os

os.chmod("llama-binaries/llama-b12542/llama-cli", 0o755)

!llama-binaries/llama-b12542/llama-cli --version

version: 1.0.0-dev (build 12542, commit 51a77c4f2)
built with GNU 13.3.0 for Linux x86_64


In [15]:
!find llama-binaries -type f | grep -E 'ggml-cuda|libggml'

llama-binaries/llama-b12542/libggml.so.1.0.0
llama-binaries/llama-b12542/libggml-cpu-haswell.so
llama-binaries/llama-b12542/libggml-base.so.1.0.0
llama-binaries/llama-b12542/libggml-cpu-skylakex.so
llama-binaries/llama-b12542/libggml-cpu-zen4.so
llama-binaries/llama-b12542/libggml-cpu-icelake.so
llama-binaries/llama-b12542/libggml-cpu-cooperlake.so
llama-binaries/llama-b12542/libggml-cpu-alderlake.so
llama-binaries/llama-b12542/libggml-cpu-sse42.so
llama-binaries/llama-b12542/libggml-cpu-cascadelake.so
llama-binaries/llama-b12542/libggml-cpu-ivybridge.so
llama-binaries/llama-b12542/libggml-rpc.so
llama-binaries/llama-b12542/libggml-cpu-piledriver.so
llama-binaries/llama-b12542/libggml-cpu-x64.so
llama-binaries/llama-b12542/libggml-cpu-sandybridge.so
llama-binaries/llama-b12542/libggml-cpu-sapphirerapids.so
llama-binaries/llama-b12542/libggml-cuda.so
llama-binaries/llama-b12542/libggml-cpu-cannonlake.so


In [16]:
import os
import subprocess

binary_dir = os.path.abspath("llama-binaries/llama-b12542")
cuda_dir = os.path.abspath(
    "llama-cpp/cudart-llama-b12542-bin-ubuntu-cuda-12.8-x64"
)

os.environ["LD_LIBRARY_PATH"] = (
    binary_dir + ":" + cuda_dir + ":" +
    os.environ.get("LD_LIBRARY_PATH", "")
)

print("llama.cpp version:")
subprocess.run([binary_dir + "/llama-cli", "--version"])

print("\nAvailable devices:")
subprocess.run([binary_dir + "/llama-cli", "--list-devices"])

llama.cpp version:

Available devices:


CompletedProcess(args=['/content/llama-binaries/llama-b12542/llama-cli', '--list-devices'], returncode=0)

In [17]:
import os
import subprocess

binary = os.path.abspath(
    "llama-binaries/llama-b12542/llama-cli"
)

result = subprocess.run(
    [binary, "--list-devices"],
    capture_output=True,
    text=True,
    env=os.environ.copy()
)

print("Return code:", result.returncode)
print("\nSTDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

Return code: 0

STDOUT:
Available devices:
  CUDA0: Tesla T4 (14912 MiB, 14807 MiB free)


STDERR:



In [18]:
from huggingface_hub import HfApi

api = HfApi()
repo = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"

files = api.list_repo_files(repo)

for file in files:
    if file.endswith(".gguf"):
        print(file)

qwen2.5-0.5b-instruct-fp16.gguf
qwen2.5-0.5b-instruct-q2_k.gguf
qwen2.5-0.5b-instruct-q3_k_m.gguf
qwen2.5-0.5b-instruct-q4_0.gguf
qwen2.5-0.5b-instruct-q4_k_m.gguf
qwen2.5-0.5b-instruct-q5_0.gguf
qwen2.5-0.5b-instruct-q5_k_m.gguf
qwen2.5-0.5b-instruct-q6_k.gguf
qwen2.5-0.5b-instruct-q8_0.gguf


In [19]:
from huggingface_hub import hf_hub_download
import os

repo = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"

q4_path = hf_hub_download(
    repo_id=repo,
    filename="qwen2.5-0.5b-instruct-q4_k_m.gguf"
)

print("Q4 model downloaded!")
print("Location:", q4_path)
print("Size (MB):", round(os.path.getsize(q4_path) / (1024**2), 2))

qwen2.5-0.5b-instruct-q4_k_m.gguf: reconstructing file:   0%|          |  0.00B /  491MB            

qwen2.5-0.5b-instruct-q4_k_m.gguf: downloading bytes:           |  0.00B            

Q4 model downloaded!
Location: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q4_k_m.gguf
Size (MB): 468.64


In [20]:
q8_path = hf_hub_download(
    repo_id=repo,
    filename="qwen2.5-0.5b-instruct-q8_0.gguf"
)

print("Q8 model downloaded!")
print("Location:", q8_path)
print("Size (MB):", round(os.path.getsize(q8_path) / (1024**2), 2))

qwen2.5-0.5b-instruct-q8_0.gguf: reconstructing file:   0%|          |  0.00B /  676MB            

qwen2.5-0.5b-instruct-q8_0.gguf: downloading bytes:           |  0.00B            

Q8 model downloaded!
Location: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf
Size (MB): 644.41


In [22]:
import subprocess

print("GPU STATUS:")
gpu = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
print(gpu.stdout)

print("\nLLAMA.CPP OPTIONS:")
help_result = subprocess.run(
    [binary, "--help"],
    capture_output=True,
    text=True,
    timeout=30
)

for line in help_result.stdout.splitlines():
    if any(word in line.lower() for word in
           ["conversation", "interactive", "prompt", "batch", "no-display"]):
        print(line)

GPU STATUS:
Thu Nov 19 16:25:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------

In [23]:
result = subprocess.run(
    [
        binary,
        "-m", q4_path,
        "-ngl", "99",
        "-c", "2048",
        "-n", "32",
        "-t", "4",
        "--temp", "0",
        "--single-turn",
        "--no-display-prompt",
        "-p", prompt
    ],
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", result.returncode)
print("\nMODEL RESPONSE:")
print(result.stdout[-2000:])
print("\nPERFORMANCE LOG:")
print(result.stderr[-3000:])

Return code: 0

MODEL RESPONSE:


Loading model... |-\|/-\|/-\|/-\ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b12542-51a77c4f2
model      : /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q4_k_m.gguf
ftype      : Q4_K - Medium
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> <|im_start|>system
You are a helpful assistant. Answer clearly and accurately.
<|im_end|>
<|im_start|>user
Explain quantization in machine learning in three simple senten

In [24]:
import subprocess
import time

cmd = [
    binary, "-m", q4_path,
    "-ngl", "99",
    "-c", "2048",
    "-n", "128",
    "-t", "4",
    "--temp", "0",
    "--single-turn",
    "--no-display-prompt",
    "-p", prompt
]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

peak_memory = 0

while process.poll() is None:
    check = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    )
    try:
        memory = int(check.stdout.strip().splitlines()[0])
        peak_memory = max(peak_memory, memory)
    except:
        pass
    time.sleep(0.5)

output, _ = process.communicate()

print("Peak GPU memory:", peak_memory, "MiB")
print("\nLLAMA OUTPUT:")
print(output[-2500:])

Peak GPU memory: 557 MiB

LLAMA OUTPUT:


Loading model... |-\|/-\|/-\|/-\ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b12542-51a77c4f2
model      : /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q4_k_m.gguf
ftype      : Q4_K - Medium
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> <|im_start|>system
You are a helpful assistant. Answer clearly and accurately.
<|im_end|>
<|im_start|>user
Explain quantization in machine learning in three simpl

In [25]:
prompts = [
    "Explain quantization in machine learning in three simple sentences.",
    "What is the difference between supervised and unsupervised learning?",
    "Calculate 25 multiplied by 16. Show the answer.",
    "Give three practical applications of natural language processing.",
    "Explain overfitting in simple terms and give one way to reduce it.",
    "Summarize the purpose of a database in two sentences.",
    "Write a Python function that returns the square of a number.",
    "What are the main differences between RAM and storage?",
    "List three advantages of using renewable energy.",
    "A customer says their headphones battery drains quickly and requests a repair. Extract product, issue_type, and requested_action as JSON."
]

print("Number of evaluation prompts:", len(prompts))
for i, p in enumerate(prompts, 1):
    print(f"{i}. {p}")

Number of evaluation prompts: 10
1. Explain quantization in machine learning in three simple sentences.
2. What is the difference between supervised and unsupervised learning?
3. Calculate 25 multiplied by 16. Show the answer.
4. Give three practical applications of natural language processing.
5. Explain overfitting in simple terms and give one way to reduce it.
6. Summarize the purpose of a database in two sentences.
7. Write a Python function that returns the square of a number.
8. What are the main differences between RAM and storage?
9. List three advantages of using renewable energy.
10. A customer says their headphones battery drains quickly and requests a repair. Extract product, issue_type, and requested_action as JSON.


In [26]:
import re
import time

q4_results = []

for i, question in enumerate(prompts, 1):
    formatted_prompt = f"""<|im_start|>system
You are a helpful assistant. Follow the user's instructions carefully.
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
"""

    result = subprocess.run(
        [
            binary, "-m", q4_path,
            "-ngl", "99",
            "-c", "2048",
            "-n", "128",
            "-t", "4",
            "--temp", "0",
            "--single-turn",
            "--no-display-prompt",
            "-p", formatted_prompt
        ],
        capture_output=True,
        text=True,
        timeout=120
    )

    output = result.stdout

    match = re.search(
        r"Generation:\s*([\d.]+)\s*t/s",
        output
    )

    speed = float(match.group(1)) if match else None

    response = output.split("> <|im_start|>assistant")[-1]
    response = re.sub(r"\[ Prompt:.*", "", response, flags=re.S).strip()
    response = response.replace("Exiting...", "").strip()

    q4_results.append({
        "prompt": question,
        "response": response,
        "tokens_per_sec": speed,
        "return_code": result.returncode
    })

    print(f"\nPrompt {i}/10")
    print("Generation speed:", speed)
    print("Response:", response[:500])


Prompt 1/10
Generation speed: 174.0
Response: Loading model... |-\|/-\|/-\|/-\ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b12542-51a77c4f2
model      : /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q4_k_m.gguf
ftyp

Prompt 2/10
Generation speed: 240.4
Response: Loading model... |-\|/-\|/-\ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b12542-51a77c4f2
model      : /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/sn

In [27]:
print("Q4 PROMPT 1 — RAW OUTPUT (LAST 1500 CHARACTERS)")
print(q4_results[0]["response"][-1500:])

Q4 PROMPT 1 — RAW OUTPUT (LAST 1500 CHARACTERS)
Loading model... |-\|/-\|/-\|/-\ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b12542-51a77c4f2
model      : /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q4_k_m.gguf
ftype      : Q4_K - Medium
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> <|im_start|>system
You are a helpful assistant. Follow the user's instructions carefully.
<|im_end|>
<|im_start|>user
Explain quantization in machine learni

In [28]:
import re

for i, item in enumerate(q4_results, 1):
    raw = item["response"]

    match = re.search(
        r"<\|im_start\|>assistant\s*(.*)",
        raw,
        flags=re.DOTALL
    )

    if match:
        answer = match.group(1).strip()
        answer = re.sub(r"\[ Prompt:.*", "", answer, flags=re.DOTALL)
        item["response"] = answer.strip()

print("Corrected Q4 responses:\n")

for i, item in enumerate(q4_results, 1):
    print(f"\nPrompt {i}:")
    print(item["response"][:600])

Corrected Q4 responses:


Prompt 1:
Quantization in machine learning is a technique used to reduce the size of a model's parameters, making it more efficient and faster to train. It involves converting a large number of parameters into a smaller number of parameters, typically by using a method called quantization. This process helps in reducing the computational complexity of the model, making it more suitable for real-world applications.

Prompt 2:
Supervised learning and unsupervised learning are two main types of machine learning algorithms used for different purposes. Here's a detailed comparison:

1. Supervised Learning:
   - **Objective**: The goal of supervised learning is to learn a function that maps inputs to outputs. The algorithm is trained on a labeled dataset, where each input is associated with a specific output.
   - **Types**: Commonly used algorithms include:
     - **Linear Regression**: Used for predicting continuous outputs based on one or more independent variabl

In [29]:
for i, item in enumerate(q4_results, 1):
    answer = item["response"]
    print(f"\nPrompt {i}")
    print("Characters:", len(answer))
    print("Ending:", answer[-180:].replace("\n", " "))
    print("Speed:", item["tokens_per_sec"], "tokens/sec")



Prompt 1
Characters: 407
Ending: ers, typically by using a method called quantization. This process helps in reducing the computational complexity of the model, making it more suitable for real-world applications.
Speed: 174.0 tokens/sec

Prompt 2
Characters: 693
Ending:  one or more independent variables.      - **Logistic Regression**: Used for binary classification problems.      - **Decision Trees**: Used for classification and regression tasks
Speed: 240.4 tokens/sec

Prompt 3
Characters: 346
Ending:  First, multiply 25 by 10 (since 16 is 10 more than 6).    - Then, add the result to 25.     Calculation:    \[    25 \times 16 = 25 \times (10 + 6)    \]    \[    = 25 \times 10 +
Speed: 239.6 tokens/sec

Prompt 4
Characters: 665
Ending: ormation, making it easier to find the most important points in a longer text.  2. **Sentiment Analysis**: Sentiment analysis is a technique used to determine the emotional tone of
Speed: 239.2 tokens/sec

Prompt 5
Characters: 645
Ending: arning the

In [30]:
q4_results_256 = []

for i, question in enumerate(prompts, 1):
    formatted_prompt = f"""<|im_start|>system
You are a helpful assistant. Follow the user's instructions carefully.
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
"""

    result = subprocess.run(
        [
            binary, "-m", q4_path,
            "-ngl", "99",
            "-c", "2048",
            "-n", "256",
            "-t", "4",
            "--temp", "0",
            "--single-turn",
            "--no-display-prompt",
            "-p", formatted_prompt
        ],
        capture_output=True,
        text=True,
        timeout=180
    )

    match = re.search(
        r"<\|im_start\|>assistant\s*(.*)",
        result.stdout,
        flags=re.DOTALL
    )

    answer = match.group(1).strip() if match else result.stdout
    answer = re.sub(r"\[ Prompt:.*", "", answer, flags=re.DOTALL)
    answer = answer.replace("Exiting...", "").strip()

    speed_match = re.search(
        r"Generation:\s*([\d.]+)\s*t/s",
        result.stdout
    )

    q4_results_256.append({
        "prompt": question,
        "response": answer,
        "tokens_per_sec": float(speed_match.group(1)) if speed_match else None,
        "return_code": result.returncode
    })

    print(
        f"Prompt {i}/10 | "
        f"Speed: {q4_results_256[-1]['tokens_per_sec']} tokens/sec | "
        f"Characters: {len(answer)}"
    )

Prompt 1/10 | Speed: 175.9 tokens/sec | Characters: 407
Prompt 2/10 | Speed: 244.5 tokens/sec | Characters: 1335
Prompt 3/10 | Speed: 240.7 tokens/sec | Characters: 482
Prompt 4/10 | Speed: 243.6 tokens/sec | Characters: 1360
Prompt 5/10 | Speed: 226.0 tokens/sec | Characters: 1295
Prompt 6/10 | Speed: 147.5 tokens/sec | Characters: 133
Prompt 7/10 | Speed: 212.7 tokens/sec | Characters: 377
Prompt 8/10 | Speed: 243.8 tokens/sec | Characters: 1219
Prompt 9/10 | Speed: 234.7 tokens/sec | Characters: 683
Prompt 10/10 | Speed: 203.5 tokens/sec | Characters: 106


In [31]:
for i in [2, 3, 6, 8, 10]:
    item = q4_results_256[i - 1]
    print(f"\n{'='*50}")
    print(f"PROMPT {i}")
    print(item["response"])


PROMPT 2
Supervised learning and unsupervised learning are two main types of machine learning algorithms used for different purposes. Here's a detailed comparison:

1. Supervised Learning:
   - **Objective**: The goal of supervised learning is to learn a function that maps inputs to outputs. The algorithm is trained on a labeled dataset, where each input is associated with a specific output.
   - **Types**: Commonly used algorithms include:
     - **Linear Regression**: Used for predicting continuous outputs based on one or more independent variables.
     - **Logistic Regression**: Used for binary classification problems.
     - **Decision Trees**: Used for classification and regression tasks.
     - **Random Forest**: Used for classification and regression tasks.
     - **Support Vector Machines (SVM)**: Used for classification and regression tasks.
     - **Neural Networks**: Used for both classification and regression tasks.
   - **Advantages**: Can handle complex relationships be

In [32]:
q8_results = []

for i, question in enumerate(prompts, 1):
    formatted_prompt = f"""<|im_start|>system
You are a helpful assistant. Follow the user's instructions carefully.
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
"""

    result = subprocess.run(
        [
            binary, "-m", q8_path,
            "-ngl", "99",
            "-c", "2048",
            "-n", "256",
            "-t", "4",
            "--temp", "0",
            "--single-turn",
            "--no-display-prompt",
            "-p", formatted_prompt
        ],
        capture_output=True,
        text=True,
        timeout=180
    )

    match = re.search(
        r"<\|im_start\|>assistant\s*(.*)",
        result.stdout,
        flags=re.DOTALL
    )

    answer = match.group(1).strip() if match else result.stdout
    answer = re.sub(r"\[ Prompt:.*", "", answer, flags=re.DOTALL)
    answer = answer.replace("Exiting...", "").strip()

    speed_match = re.search(
        r"Generation:\s*([\d.]+)\s*t/s",
        result.stdout
    )

    q8_results.append({
        "prompt": question,
        "response": answer,
        "tokens_per_sec": float(speed_match.group(1)) if speed_match else None,
        "return_code": result.returncode
    })

    print(
        f"Prompt {i}/10 | "
        f"Speed: {q8_results[-1]['tokens_per_sec']} tokens/sec | "
        f"Characters: {len(answer)}"
    )

Prompt 1/10 | Speed: 173.9 tokens/sec | Characters: 443
Prompt 2/10 | Speed: 226.1 tokens/sec | Characters: 1297
Prompt 3/10 | Speed: 187.9 tokens/sec | Characters: 303
Prompt 4/10 | Speed: 227.3 tokens/sec | Characters: 1092
Prompt 5/10 | Speed: 221.8 tokens/sec | Characters: 639
Prompt 6/10 | Speed: 177.2 tokens/sec | Characters: 143
Prompt 7/10 | Speed: 226.6 tokens/sec | Characters: 848
Prompt 8/10 | Speed: 226.1 tokens/sec | Characters: 1229
Prompt 9/10 | Speed: 197.1 tokens/sec | Characters: 690
Prompt 10/10 | Speed: 191.4 tokens/sec | Characters: 106


In [33]:
for i in [2, 3, 6, 8, 10]:
    print(f"\n{'='*60}")
    print(f"Q8 — PROMPT {i}")
    print(f"Question: {q8_results[i-1]['prompt']}")
    print(f"Response:\n{q8_results[i-1]['response']}")


Q8 — PROMPT 2
Question: What is the difference between supervised and unsupervised learning?
Response:
Supervised learning and unsupervised learning are two main types of machine learning algorithms used for various tasks. Here's a detailed comparison between the two:

1. Supervised Learning:
   - Supervised learning involves training a model on a labeled dataset, where the input data (features) and the corresponding output (target) are provided.
   - The model learns to predict the target values based on the input features.
   - The goal is to learn a mapping from input features to target values.
   - Examples of supervised learning include:
     - Classification: predicting the class label for each data point.
     - Regression: predicting a continuous value (e.g., temperature) for each data point.
     - Clustering: grouping similar data points together.

2. Unsupervised Learning:
   - Unsupervised learning involves training a model on an unlabeled dataset, where the input data is 

In [34]:
import subprocess
import time
import re

def measure_gpu_memory(model_path, model_name):
    formatted_prompt = f"""<|im_start|>system
You are a helpful assistant. Follow the user's instructions carefully.
<|im_end|>
<|im_start|>user
{prompts[7]}
<|im_end|>
<|im_start|>assistant
"""

    process = subprocess.Popen(
        [
            binary, "-m", model_path,
            "-ngl", "99",
            "-c", "2048",
            "-n", "256",
            "-t", "4",
            "--temp", "0",
            "--single-turn",
            "--no-display-prompt",
            "-p", formatted_prompt
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    memory_samples = []

    while process.poll() is None:
        try:
            memory = subprocess.check_output(
                [
                    "nvidia-smi",
                    "--query-gpu=memory.used",
                    "--format=csv,noheader,nounits"
                ],
                text=True
            )
            memory_samples.append(int(memory.strip().split("\n")[0]))
        except Exception:
            pass

        time.sleep(0.1)

    stdout, stderr = process.communicate()

    speed_match = re.search(
        r"Generation:\s*([\d.]+)\s*t/s",
        stdout
    )

    print(f"\n{model_name}")
    print("Peak GPU memory:", max(memory_samples), "MiB")
    print(
        "Generation speed:",
        speed_match.group(1) if speed_match else "Not captured",
        "tokens/sec"
    )

measure_gpu_memory(q4_path, "Q4_K_M")
measure_gpu_memory(q8_path, "Q8_0")


Q4_K_M
Peak GPU memory: 557 MiB
Generation speed: 218.8 tokens/sec

Q8_0
Peak GPU memory: 685 MiB
Generation speed: 224.2 tokens/sec
